##  Import

In [ ]:
import keras
import numpy as np
import os
import time

learning_rate = 0.05


(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

x_train_mean = np.mean(x_train)
x_train_std = np.std(x_train)

x_train = (x_train - x_train_mean) / x_train_std
x_test = (x_test - x_train_mean) / x_train_std

y_train = keras.utils.to_categorical(y_train, 10) * 2.0 - 1.0
y_test = keras.utils.to_categorical(y_test, 10) * 2.0 - 1.0

for seed in [2, 7, 24, 18, 42]:
    keras.utils.set_random_seed(seed)
    model = keras.models.Sequential()
    model.add(keras.layers.Flatten())
    model.add(keras.layers.Dense(32, keras.activations.tanh))
    model.add(keras.layers.Dense(16, keras.activations.tanh))
    model.add(keras.layers.Dense(10, keras.activations.tanh))

    model.compile(loss=keras.losses.mean_squared_error,
                  optimizer=keras.optimizers.SGD(learning_rate=learning_rate),
                  metrics=[keras.metrics.categorical_accuracy]
                  )

        # start_time = time.time()
        # for _ in range(10):
        #     _ = model.predict(x_train, batch_size=512)
        # print(f"Elapsed time : {(time.time() - start_time) / 10.0}")

    model.fit(x_train, y_train,
              validation_data=(x_test, y_test),
              epochs=10,
              batch_size=256)


### Clustering K-means

In [29]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from pyspark.sql import SparkSession
from pyspark.sql.functions import when, col

from sklearn.decomposition import PCA

#---Récupération des données---
spark = (
    SparkSession.builder
    .master("local[*]")
    .getOrCreate()
)

data = spark.read.csv("../../../datasets/gaming_mental_health/gaming_mental_health.csv",
                      header = True,
                      inferSchema=True)


'''
numerical_columns = [
    'age', 'sleep_hours', 'mood_swing_frequency',
    'withdrawal_symptoms', 'loss_of_other_interests',
    'continued_despite_problems', 'eye_strain',
    'back_neck_pain', 'exercise_hours_weekly',
    'social_isolation_score', 'face_to_face_social_hours_weekly',
    'monthly_game_spending_usd', 'years_gaming'
]'''

numerical_columns = [ 'gender', 'game_genre', 'gaming_platform', 'age', 'gaming_addiction_risk_level',
                     'sleep_hours','sleep_quality', 'sleep_disruption_frequency',
                     'mood_state', 'mood_swing_frequency',
                     'withdrawal_symptoms', 'loss_of_other_interests','continued_despite_problems', 
                     'social_isolation_score', 'face_to_face_social_hours_weekly',
                     'monthly_game_spending_usd', 'years_gaming', 
                     'grades_gpa','exercise_hours_weekly'
                     
]

print("Count de data : ", data.count())
#data[numerical_columns].show()


#---Transformation des données---
 

'''data = data.withColumn(
    "gender",
    when(col("gender") == "Male", -1)
    .when(col("gender") == "Other", 0)
    .when(col("gender") == "Female", 1)
    .otherwise(None)
)'''

data = data.withColumn(
    "sleep_quality",
    when(col("sleep_quality") == "Insomnia", -1)
    .when(col("sleep_quality") == "Very Poor", -0.5)
    .when(col("sleep_quality") == "Poor", 0)
    .when(col("sleep_quality") == "Fair", 0.5)
    .when(col("sleep_quality") == "Good", 1)
    .otherwise(None)
)

data = data.withColumn(
    "sleep_disruption_frequency",
    when(col("sleep_disruption_frequency") == "Always", -1)
    .when(col("sleep_disruption_frequency") == "Often", -0.5)
    .when(col("sleep_disruption_frequency") == "Sometimes", 0)
    .when(col("sleep_disruption_frequency") == "Rarely", 0.5)
    .when(col("sleep_disruption_frequency") == "Never", 1)
    .otherwise(None)
)

data = data.withColumn(
    "mood_swing_frequency",
    when(col("mood_swing_frequency") == "Daily", -1)
    .when(col("mood_swing_frequency") == "Often", -0.5)
    .when(col("mood_swing_frequency") == "Sometimes", 0)
    .when(col("mood_swing_frequency") == "Rarely", 0.5)
    .when(col("mood_swing_frequency") == "Never", 1)
    .otherwise(None)
)

data = data.withColumn(
    "withdrawal_symptoms",
    when(col("withdrawal_symptoms") == "FALSE", 0)
    .when(col("withdrawal_symptoms") == "TRUE", 1)
    .otherwise(None)
)

data = data.withColumn(
    "loss_of_other_interests",
    when(col("loss_of_other_interests") == "FALSE", 0)
    .when(col("loss_of_other_interests") == "TRUE", 1)
    .otherwise(None)
)

data = data.withColumn(
    "gaming_addiction_risk_level",
    when(col("gaming_addiction_risk_level") == "Severe", -1)
    .when(col("gaming_addiction_risk_level") == "High", -0.5)
    .when(col("gaming_addiction_risk_level") == "Moderate", 0.5)
    .when(col("gaming_addiction_risk_level") == "Low", 1)
    .otherwise(None)
)

data = data.withColumn(
    "continued_despite_problems",
    when(col("continued_despite_problems") == "FALSE", 0)
    .when(col("continued_despite_problems") == "TRUE", 1)
    .otherwise(None)
)


data_filtre = data[numerical_columns].dropna().toPandas()
data_filtre = pd.get_dummies(data_filtre, columns=['gender', 'game_genre', 'gaming_platform', 'mood_state'])



#--- ---Standardisation des data + création des cluster --- ---
scaler = StandardScaler()
data_scale = scaler.fit_transform(data_filtre)

#--Valeur du K-Menas--
nb_cluster = 13
seed = 74


kmeans = KMeans(n_clusters=nb_cluster, random_state=seed)
cluster_data_f = kmeans.fit_predict(data_scale)



data_filtre['Cluster'] = cluster_data_f
#--Vérification du fonctionnement--
#print(data_filtre.columns)
#print(data_filtre.groupby('Cluster').mean())



pca = PCA(n_components=2)
X_pca = pca.fit_transform(data_scale)

'''
#---Visuel---
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_data_f)
plt.xlabel('Composante 1')
plt.ylabel('Composante 2')
plt.title('Clusters applati en 2D')
plt.show()
'''


#--Enregistrement des cluster en txt---
with open(f"../donnee/moyennes_cluster{nb_cluster}_seed{seed}.txt", 'w') as f:
    f.write(data_filtre
            .groupby('Cluster')
            .mean()
            .sort_values(by='gaming_addiction_risk_level', ascending=False)
            .to_string())


Count de data :  1000


In [74]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .getOrCreate()
)

df = spark.read.csv("../../../datasets/gaming_mental_health/gaming_mental_health.csv",
    header=True,
    inferSchema=True
)

df.dropna().show(20)

+---------+---+------+------------------+-------------+-----------------+---------------+-----------+-------------+--------------------------+-------------------------+----------+-----------------------+----------+--------------------+-------------------+-----------------------+--------------------------+----------+--------------+----------------+---------------------+----------------------+--------------------------------+-------------------------+------------+---------------------------+
|record_id|age|gender|daily_gaming_hours|   game_genre|     primary_game|gaming_platform|sleep_hours|sleep_quality|sleep_disruption_frequency|academic_work_performance|grades_gpa|work_productivity_score|mood_state|mood_swing_frequency|withdrawal_symptoms|loss_of_other_interests|continued_despite_problems|eye_strain|back_neck_pain|weight_change_kg|exercise_hours_weekly|social_isolation_score|face_to_face_social_hours_weekly|monthly_game_spending_usd|years_gaming|gaming_addiction_risk_level|
+---------